# 05: Monolithic Student: walk-pick BC distillation plus BeyondMimic-style PPO fine-tune

| | |
|---|---|
| **Project**    | Humanoid Loco-Manipulation with the Unitree G1 |
| **Developer**  | Narayane Ribeiro Medeiros |
| **Repository** | [https://github.com/narayanerm/unitree-g1-humanoid-monolithic-policy](https://github.com/narayanerm/unitree-g1-humanoid-monolithic-policy) |

Distills the walk + pick FSM into a single goal-conditioned MLP (115 obs, 29 joint targets), then fine-tunes with PPO against the pick task reward using a frozen BC prior as mimic reward and KL anchor.

### Run-Mode Flags

`REUSE_DATASET` skips the FSM data-gen; `REUSE_BC` skips both the dataset and the BC training (implies `REUSE_DATASET=True`); PPO always runs. `DEBUG_LIGHT` shrinks everything to a 2-3 min sanity pass. `USE_ASTAR` picks A* vs reactive pure-pursuit as the walk-phase steering source.

In [ ]:
DEBUG_LIGHT     = False   # True: 4 eps + 3 epochs (~2 min end-to-end sanity)
REUSE_DATASET   = False   # skip FSM data-gen if outputs/<DATASET_NAME> already exists
REUSE_BC        = False   # skip BC and reuse latest g1_monolithic checkpoint; implies REUSE_DATASET=True
USE_ASTAR       = True    # walk phase follows an A* path around the pick table
WORKERS         = 1 if DEBUG_LIGHT else 20   # parallel subprocess pool for the data-gen; num_envs=1 per worker (mjwarp batch divergence)

K_EPISODES      = 4 if DEBUG_LIGHT else 2000
BASE_SEED       = 100
SPAWN_R_MIN     = 0.2     # min robot spawn radius (m) around the table
SPAWN_R_MAX     = 5.0     # max robot spawn radius (m); initial yaw uniform in [-pi, pi)
MAX_STEPS       = 1200    # per-episode step cap

if REUSE_BC and not REUSE_DATASET:
    print("[config] REUSE_BC=True implies REUSE_DATASET=True; promoting.")
    REUSE_DATASET = True

DATASET_NAME = "walk_pick_astar_sequential.npz" if USE_ASTAR else "walk_pick_sequential.npz"
FAILED_DIR   = "outputs/failed_episodes"

### Environment Setup

Enforces Python 3.10 and sets headless MuJoCo (`MUJOCO_GL=egl`) before any mjlab import so the notebook runs on a remote GPU without a display.

In [ ]:
import os, sys
assert sys.version_info[:2] == (3, 10), f"Python 3.10 required, got {sys.version_info[:2]}"
os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
print(f"cwd={os.getcwd()}")

### Dataset Generation

Runs `src.fsm_continuous_sequential` with `WORKERS` parallel subprocesses, each with `num_envs=1`. With `USE_ASTAR=True` each episode plans an 8-connected A* path from the robot spawn to the nearest table edge, then the walk expert pure-pursuits the waypoints and brakes into the handover envelope. Writes `outputs/<DATASET_NAME>` with `unified_obs`, `action`, `phase` arrays.

In [ ]:
import subprocess
from pathlib import Path

DATASET_LOCAL = Path(f"outputs/{DATASET_NAME}").resolve()
REPO_ROOT     = Path(os.getcwd()).resolve()

if REUSE_DATASET and DATASET_LOCAL.exists():
    print(f"[data] reusing existing {DATASET_LOCAL}")
else:
    tag = "A*-guided" if USE_ASTAR else "reactive pure-pursuit"
    print(f"[data] running sequential FSM builder ({tag}, {K_EPISODES} eps, workers={WORKERS})")
    cmd = [
        sys.executable, "-m", "src.fsm_continuous_sequential",
        "--episodes", str(K_EPISODES),
        "--base-seed", str(BASE_SEED),
        "--max-steps", str(MAX_STEPS),
        "--r-min", str(SPAWN_R_MIN),
        "--r-max", str(SPAWN_R_MAX),
        "--out", str(DATASET_LOCAL),
        "--failed-dir", FAILED_DIR,
        "--workers", str(WORKERS),
    ]
    if USE_ASTAR:
        cmd += ["--astar"]
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), env={**os.environ, "MUJOCO_GL": "egl"})
    if r.returncode != 0:
        raise RuntimeError(f"Sequential FSM builder failed with exit {r.returncode}")
    assert DATASET_LOCAL.exists(), f"builder returned 0 but {DATASET_LOCAL} was not written"

import numpy as np
blob = np.load(str(DATASET_LOCAL))
obs, act, phase = blob["unified_obs"], blob["action"], blob["phase"]
print(f"dataset: {obs.shape[0]} transitions, obs={obs.shape[1]}, act={act.shape[1]}")
for pid in np.unique(phase):
    name = ["walk", "pick", "carry", "place"][int(pid)]
    n_p = int((phase == pid).sum())
    print(f"  phase {int(pid)} ({name}): {n_p} transitions ({n_p / obs.shape[0] * 100:.1f}%)")

### Dataset Diagnostics

Reads `outputs/failed_episodes/metadata.json` and reports walk-vs-pick fall split, survival-step distribution, and the top-10 shortest failures as candidates for video inspection. Pure numpy/matplotlib, no mjlab.

In [ ]:
import json
from pathlib import Path
import numpy as np

meta_path = Path(FAILED_DIR) / "metadata.json"
if not meta_path.exists():
    print(f"[diag] no metadata at {meta_path}; skip (run generate cell first)")
else:
    meta = json.loads(meta_path.read_text())
    per_ep = meta["per_episode"]
    failed = [e for e in per_ep if e["dropped"]]
    kept   = [e for e in per_ep if not e["dropped"]]
    print(f"[diag] {len(failed)}/{meta['episodes']} fell ({100*len(failed)/meta['episodes']:.1f}%); {len(kept)} kept")

    walk_falls = [e for e in failed if e["pick_kept"] == 0]
    pick_falls = [e for e in failed if e["pick_kept"] > 0]
    print(f"  fell during walk: {len(walk_falls)} ({100*len(walk_falls)/max(1,len(failed)):.1f}% of falls)")
    print(f"  fell during pick: {len(pick_falls)} ({100*len(pick_falls)/max(1,len(failed)):.1f}% of falls)")

    if failed:
        t = np.array([e["transitions"] for e in failed])
        print(f"  survival steps (failed eps): min={t.min()}, p25={np.percentile(t,25):.0f}, median={np.median(t):.0f}, p75={np.percentile(t,75):.0f}, max={t.max()}")
        pk = np.array([e["pick_kept"] for e in pick_falls]) if pick_falls else np.array([0])
        print(f"  steps in pick before fall (pick-falls only): min={pk.min()}, median={np.median(pk):.0f}, max={pk.max()}")

    top = sorted(failed, key=lambda e: e["transitions"])[:10]
    print("\n  10 shortest failures (candidates for video inspection):")
    print(f"    {'seed':>6}  {'total':>6}  {'walk':>6}  {'pick':>6}  npz")
    for e in top:
        npz_short = Path(e['failed_npz']).name if e.get('failed_npz') else '-'
        print(f"    {e['seed']:>6d}  {e['transitions']:>6d}  {e['walk_kept']:>6d}  {e['pick_kept']:>6d}  {npz_short}")

### BC Training

Calls `train_bc` from `src.bc_train` with phase-balanced sampling (walk dominates pick ~12:1 in the dataset). Loss is per-joint MSE on the expert action. `REUSE_BC=True` recovers the latest checkpoint from `outputs/logs/g1_monolithic/`.

In [ ]:
from src.bc_train import train_bc, BCConfig

cfg = BCConfig(
    epochs=3 if DEBUG_LIGHT else 2000,
    batch_size=256 if DEBUG_LIGHT else 8192,
    lr=6e-4 if not DEBUG_LIGHT else 3e-4,
    balance_phases=True,
)

if REUSE_BC:
    from src.utils import find_latest_checkpoint
    ck = find_latest_checkpoint("outputs/logs", experiment_hint="g1_monolithic")
    if ck is None:
        print("[train] no previous run found; training from scratch")
        ck = train_bc(DATASET_LOCAL, cfg=cfg)
    else:
        print(f"[train] reusing {ck}")
else:
    ck = train_bc(DATASET_LOCAL, cfg=cfg)

print(f"[train] BC checkpoint: {ck}")

### Held-out MSE Sanity

Loads the BC checkpoint and reports overall + per-phase MSE on the first 8192 transitions. Pick MSE typically runs 3x worse than walk because pick is the finer-grained control regime and under-represented in the dataset.

In [ ]:
import torch
from src.student_policy import MonolithicPolicyStudent

device = "cuda:0" if torch.cuda.is_available() else "cpu"
student = MonolithicPolicyStudent.load(ck, device=device)
student.eval()

n_eval = min(8192, obs.shape[0])
with torch.no_grad():
    x = torch.from_numpy(obs[:n_eval]).float().to(device)
    y = torch.from_numpy(act[:n_eval]).float().to(device)
    y_hat = student(x)
    mse_overall = torch.nn.functional.mse_loss(y_hat, y).item()
print(f"held-out BC MSE on {n_eval} transitions: {mse_overall:.5f}")

p = phase[:n_eval]
for pid in np.unique(p):
    name = ["walk", "pick", "carry", "place"][int(pid)]
    mask = p == pid
    if mask.sum() == 0:
        continue
    with torch.no_grad():
        mse_p = torch.nn.functional.mse_loss(y_hat[mask], y[mask]).item()
    print(f"  {name}: {int(mask.sum())} transitions, MSE={mse_p:.5f}")

### PPO Fine-tune (BeyondMimic-style)

Gaussian actor warm-started from the BC checkpoint with scalar `log_std` frozen for the first `freeze_log_std_iters` iterations. From-scratch MLP critic. Composite reward: pick env reward + walk-distance shaping + `-mimic_coef * MSE(a, a_bc)` + survival bonus / fall penalty. KL penalty against the frozen BC prior anchors the PPO loss. Always runs; no reuse flag.

In [ ]:
from src.ppo_finetune import PPOFinetuneConfig, ppo_finetune

ppo_cfg = PPOFinetuneConfig(
    num_envs=256 if DEBUG_LIGHT else 2048,
    num_steps_per_env=8 if DEBUG_LIGHT else 24,
    max_iterations=20 if DEBUG_LIGHT else 1500,
    episode_max_steps=200 if DEBUG_LIGHT else 400,
    spawn_r_min=SPAWN_R_MIN,
    spawn_r_max=SPAWN_R_MAX,
    freeze_log_std_iters=5 if DEBUG_LIGHT else 100,
    save_interval=5 if DEBUG_LIGHT else 25,
    device=device,
)

ppo_ck = ppo_finetune(bc_ckpt=ck, cfg=ppo_cfg)
print(f"[ppo] fine-tuned checkpoint: {ppo_ck}")

### TensorBoard

BC curves live under `outputs/logs/g1_monolithic/`; PPO curves under `outputs/logs/g1_monolithic_ppo/`. Both include `Perf/eta_s` for remaining-time projection.

In [ ]:
# Launch TensorBoard from a terminal:
#   tensorboard --logdir_spec bc:outputs/logs/g1_monolithic,ppo:outputs/logs/g1_monolithic_ppo